# Conversions Silver

---

Conversões financeiras aplicadas de usuários.

---

#### Esquema Canônico


| Campos | Tipo | Obrigatório | Regra |
| :--- | :--- | :---: | :--- |
| **conversion_id** | string | sim | Identificador |
| **user_id** | string | sim | Deve existir |
| **revenue** | decimal(10,2) | sim | >= 0 |
| **conversion_date** | date | sim | Data válida |
| **source_file** | string | sim | Rastreabilidade |


## Regras de Transformação
---

#### Tipagem
* `conversion_date` → date
* `revenue` → decimal(10, 2)

#### Validação
* Remover `revenue < 0`
* Remover `user_id` nulos

#### Deduplicação
* **Regra:** Manter o registro mais recente por `conversion_id`



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
conversions_bronze = "/Volumes/main/lakehouse_marketing/bronze/conversions/"

df_conversions = spark.read\
    .format("delta")\
    .load(conversions_bronze)

print(df_conversions.count())
display(df_conversions.limit(10))


In [0]:
df_conversions.printSchema()

In [0]:
df = df_conversions.select(
    "conversion_id",
    "user_id",
    "campaign_id",
    "conversion_date",
    "revenue",
    "ingestion_timestamp",
    "source_file"

)

################################
# Converte a data para DATE  e formato "yyyy-MM-dd". 
# A especificação em "dd-MM-yyyy" não é o formato que 
# irá assumir, e sim como vem na tabela fonte.
###############################
df_typed = df\
    .withColumn("conversion_date", F.to_date(F.col("conversion_date"), "dd-MM-yyyy"))\
    .withColumn("revenue", F.col("revenue").cast("decimal(10, 2)"))



df_with_rules = df_typed.withColumn(
    "rejection_reason",
    F.when(F.col('user_id').isNull(), 'NULL_USER_ID')
     .when(F.col('revenue') < 0, 'NEGATIVE_REVENUE')
    .otherwise(None)
)

df_with_rules.show()

In [0]:
df_valid = df_with_rules.filter(F.col('rejection_reason').isNull())
df_rejected = df_with_rules.filter(F.col('rejection_reason').isNotNull())

print(df_valid.count())
print(df_rejected.count())

* Deduplicação
---

Manter o registro mais recente por `conversion_id`

In [0]:
# Particionamento por conversion_id e ordenação por conversion_date
window = Window.partitionBy("conversion_id").orderBy(F.col("conversion_date").desc())

# Após definimos a janela, o row_number define um rank para cada registro
# como ordenamos de forma decrescente, o maior "conversion_date" de cada "user_id"
# terá o rank 1, com isso, conseguimos filtrar apenas o que for igual a  1.
df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print(f"Sem deduplicação: {df_valid.count()}")
print(f"Com deduplicação: {df_dedup.count()}")

Escrita na SILVER

In [0]:

################################################
######## ESCRITA DOS DADOS TRATADOS ############
################################################
df_valid.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('main.silver_marketing.conversions')


################################################
####### ESCRITA DOS DADOS SEM TRATAMENTO #######
################################################
df_rejected.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('main.governance_marketing.conversions_rejected')

In [0]:
%sql
SELECT COUNT(*) FROM main.silver_marketing.conversions